##L07: MERGE, schema evolution and constraints
Goal: Upsert changes with MERGE, add a column with schema evolution, and protect a table with constraints.

###Step 1. Make a copy to experiment on

In [0]:
%sql
CREATE OR REPLACE TABLE da.default.zones_copy 
AS 
SELECT * FROM da.bronze.taxi_zones
;

SELECT COUNT(*) FROM da.default.zones_copy;

###Step 2. Upsert with MERGE
One row exists (LocationID 1) and is renamed; one row is new (999).

In [0]:
%sql
MERGE INTO da.default.zones_copy AS t
USING (SELECT 1 AS LocationID, 'EWR' AS Borough, 'Newark Airport (renamed)' AS Zone, 'EWR' AS service_zone, CAST(NULL AS STRING) AS _rescued_data
       UNION ALL
       SELECT 999, 'Test', 'New Zone', 'Boro Zone', NULL) s
ON t.LocationID = s.LocationID
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
SELECT * FROM da.default.zones_copy WHERE LocationID IN (1, 999);

###Step 3. Read what MERGE did
Open operationMetrics: numTargetRowsUpdated and numTargetRowsInserted should each be 1.

In [0]:
%sql
DESCRIBE HISTORY da.default.zones_copy;

###Step 4. Delete matched rows with a condition

In [0]:
%sql
MERGE INTO da.default.zones_copy t
USING (SELECT 999 AS LocationID) s
ON t.LocationID = s.LocationID
WHEN MATCHED THEN DELETE;
SELECT COUNT(*) FROM da.default.zones_copy;

###Step 5. Schema evolution on write
Append rows that have an extra column and allow the schema to evolve.

In [0]:
from pyspark.sql import functions as F
new = (spark.table("da.default.zones_copy").limit(3).withColumn("is_airport", F.lit(True)))
(new.write.mode("append").option("mergeSchema", "true").saveAsTable("da.default.zones_copy"))
spark.table("da.default.zones_copy").printSchema()

###Step 6. Constraints
A CHECK constraint rejects bad writes; a NOT NULL constraint rejects nulls.
The last statement must fail; note the error class.

In [0]:
%sql
ALTER TABLE da.default.zones_copy ADD CONSTRAINT valid_location_id CHECK (LocationID > 0);
ALTER TABLE da.default.zones_copy ALTER COLUMN LocationID SET NOT NULL;
INSERT INTO da.default.zones_copy (LocationID, Borough, Zone, service_zone) VALUES (-5, 'Bad', 'Bad', 'Bad');

###Step 7. Clean up

In [0]:
%sql
DROP TABLE IF EXISTS da.default.zones_copy;